In [1]:
import json
import pickle

def read_json(file_path):    
    with open(file_path) as f:
        json_data = json.load(f)

    return json_data

def read_pickle(file_path):
    with open(file_path, 'rb') as f:
        pickle_data = pickle.load(f)

    return pickle_data

def read_jsonl(filepath):
    data = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data

def save_json(data, filepath, indent=2):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=indent, ensure_ascii=False)


def save_pickle(data, filepath, protocol=pickle.HIGHEST_PROTOCOL):
    with open(filepath, "wb") as f:
        pickle.dump(data, f, protocol=protocol)

def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False))
            f.write("\n")

In [3]:
population = read_json("../trainings/20260128_151114_Qwen-Qwen3-4B/population.json")

In [6]:
for k in population["population"]:
    print(k)

{'node_id': 0, 'parent_id': None, 'children_ids': [1, 2, 3, 4], 'inference_prompt': 'You are given a relation name, a description of the relation in brackets, a support sentence that exemplify the relation, and a query sentence.\n\nA relation connects the Subject and the Object entities. The Subject and the Object entities are indicated with subject and object tags, respectively. You need to decide whether the relation holds between the Subject and the Object in the query sentence.\n\nRelation name: "#RELATION#" (#RELATION_DESCRIPTION#)\n\n#SUPPORT_SENTENCE_BLOCK#\n\nQuery Sentence: #QUERY_SENTENCE#\n\nIf the relation holds between the Subject and Object in the query sentence, say "yes"; otherwise, say "no". Output only "yes" or "no", and nothing else.\n', 'feedback': '', 'val_score': {'precision_mean': 0.2757885446183862, 'precision_std': 0.04528769948434521, 'recall_mean': 0.4268590998043053, 'recall_std': 0.054340899682171276, 'f1_mean': 0.3347806687530554, 'f1_std': 0.0495406634614

In [7]:
id2node = dict()

for k in population["population"]:
    id2node[k["node_id"]] = k

In [8]:
top_score = 0
top_node = 0
for k in population["population"]:
    if k["node_id"] == 0:
        continue
    if k["val_score"]["f1_mean"] > top_score:
        top_score = k["val_score"]["f1_mean"]
        top_node = k["node_id"]

print(top_node)
print(top_score)

17
0.256175994452734


In [13]:
cur_node = id2node[5]

while (cur_node != None):
    print(f"Node ID: {cur_node['node_id']}")
    metrics = cur_node["val_score"]
    print(
        f"precision={metrics['precision_mean'] * 100:.2f}±{metrics['precision_std'] * 100:.2f}, "
        f"recall={metrics['recall_mean'] * 100:.2f}±{metrics['recall_std'] * 100:.2f}, "
        f"f1={metrics['f1_mean'] * 100:.2f}±{metrics['f1_std'] * 100:.2f}"
    )
    print("------")

    print(cur_node["inference_prompt"])
    print("------------------------------")


    cur_node = id2node.get(cur_node["parent_id"], None)

Node ID: 5
precision=11.08±2.55, recall=51.95±7.71, f1=18.23±3.92
------
You are given a relation name, a description of the relation in brackets, a support sentence that exemplifies the relation, and a query sentence.

A relation connects the Subject and the Object entities. The Subject and the Object entities are indicated with subject and object tags, respectively. You need to decide whether the relation holds between the Subject and the Object in the query sentence.

Relation name: "#RELATION#" (#RELATION_DESCRIPTION#)

#SUPPORT_SENTENCE_BLOCK#

Query Sentence: #QUERY_SENTENCE#

Key considerations:
- The relation must involve a subject (a person or entity) and an object (another person or entity) that are connected by the specified relationship.
- The relation is binary and must be explicitly or implicitly expressed in the query sentence.
- Be cautious of ambiguous or indirect references. The relation should be directly or clearly implied by the sentence.
- If the subject or object